# Differential Equations — Session 17
## Section 4.5: Undetermined Coefficients — Annihilator Approach

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives


1. Define differential operators and annihilators.
2. factor constant-coefficient operators.
3. identify minimal annihilators for common forcing functions.
4. use root multiplicities to infer a trial particular solution.
5. connect annihilators with the multiplication rule.


> The theoretical sequence follows the supplied section, while all examples, diagrams, and simulations are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |\n|---:|---|\n| 0–18 min | Operators, factorization, commutation |\n| 18–40 min | Annihilator catalog |\n| 40–62 min | Higher-order homogeneous equation |\n| 62–78 min | Deleting complementary terms |\n| 78–88 min | Symbolic verification |\n| 88–90 min | Exit check |

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.linalg import expm
from IPython.display import display, Markdown
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE=True
except ImportError:
    WIDGETS_AVAILABLE=False

if os.environ.get('NB_VALIDATION_MODE') == '1':
    WIDGETS_AVAILABLE=False
plt.rcParams['figure.figsize']=(8,5)
plt.rcParams['axes.grid']=True
np.set_printoptions(precision=6,suppress=True)

def solve_second_order(f, span, y0, yp0, points=900, **kwargs):
    def rhs(x,z): return [z[1], f(x,z[0],z[1])]
    t=np.linspace(span[0],span[1],points)
    return solve_ivp(rhs,span,[y0,yp0],t_eval=t,**kwargs)

def wronskian_symbolic(funcs,x):
    return sp.simplify(sp.Matrix([[sp.diff(f,x,j) for f in funcs] for j in range(len(funcs))]).det())

print('Notebook ready.')
print('Interactive widgets available:',WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 4.5-A — Annihilator

A constant-coefficient differential operator $A(D)$ annihilates $f$ if

$$
A(D)f=0.
$$

### Catalog

- $D^{n+1}$ annihilates every polynomial of degree at most $n$.
- $(D-a)^k$ annihilates $e^{ax}$ times every polynomial of degree at most $k-1$.
- $[(D-a)^2+b^2]^k$ annihilates $e^{ax}$ times polynomial-degree $k-1$ combinations of $\cos bx$ and $\sin bx$.

### Annihilator method

For $L(D)y=g$:

1. find $y_c$ from $L(D)y=0$;
2. choose $A(D)$ annihilating $g$;
3. solve $A(D)L(D)y=0$ formally;
4. remove terms belonging to $y_c$; the remaining basis gives the form of $y_p$;
5. substitute into the original equation to determine coefficients.

### Classroom Checkpoint — Choose an Annihilator

Give an annihilator for

$$
x^2e^{3x}.
$$

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Symbolic annihilation

In [ ]:
x=sp.symbols('x',real=True)
def D(expr,n=1): return sp.diff(expr,x,n)
examples=[('quadratic',x**2+3*x+1,D(x**2+3*x+1,3)),('exp poly',x*sp.exp(2*x),sp.expand((D(D(x*sp.exp(2*x))-2*x*sp.exp(2*x))-2*(D(x*sp.exp(2*x))-2*x*sp.exp(2*x))))),('trig',sp.cos(3*x),D(sp.cos(3*x),2)+9*sp.cos(3*x))]
for name,f,r in examples: print(name); display(sp.simplify(r))

## 2. Root multiset viewpoint

The forcing $x^2e^{x}\cos3x$ is annihilated by

$$
[(D-1)^2+9]^3.
$$

Its trial space is generated by

$$
e^x P_2(x)\cos3x,\qquad e^xQ_2(x)\sin3x.
$$

In [ ]:
def trial_dimension(poly_degree=2,trig=True):
    dim=2*(poly_degree+1) if trig else poly_degree+1
    print('trial-space dimension:',dim)
if WIDGETS_AVAILABLE: interact(trial_dimension,poly_degree=IntSlider(min=0,max=5,step=1,value=2),trig=Dropdown(options=[True,False],value=True))
else: trial_dimension()

## 3. Multiplicity automatically handles duplication

Suppose

$$
(D^2+4)y=\cos2x.
$$

The forcing annihilator is also $D^2+4$. The combined operator has $(D^2+4)^2$, so the extra solutions are

$$
x\cos2x,\qquad x\sin2x.
$$

This is exactly the multiplication rule from Section 4.4.

In [ ]:
m=sp.symbols('m'); display(sp.factor((m**2+4)**2))

## 4. Complete example

For

$$
y''+3y'+2y=4x^2,
$$

$D^3$ annihilates the forcing. The combined roots are $-1,-2,0,0,0$. Deleting $e^{-x},e^{-2x}$ leaves the trial $A+Bx+Cx^2$.

In [ ]:
x=sp.symbols('x'); A,B,C=sp.symbols('A B C'); yp=A+B*x+C*x**2
coeff=sp.Poly(sp.expand(sp.diff(yp,x,2)+3*sp.diff(yp,x)+2*yp-4*x**2),x).all_coeffs(); display(sp.solve(coeff,[A,B,C],dict=True))

## Interactive exploration — Function families and annihilators

Choose a forcing family. The plot shows the function; the symbolic output verifies that the stated operator sends it to zero.

In [ ]:
def annihilator_explorer(kind='polynomial'):
    x=sp.symbols('x',real=True)
    if kind=='polynomial':
        f=x**3-2*x+1; residual=sp.diff(f,x,4); label=r'$D^4$'
    elif kind=='exponential':
        f=(1+x)*sp.exp(2*x); residual=sp.expand((sp.diff(sp.diff(f,x)-2*f,x)-2*(sp.diff(f,x)-2*f))); label=r'$(D-2)^2$'
    else:
        f=sp.exp(-x)*sp.cos(3*x); residual=sp.simplify(sp.diff(f,x,2)+2*sp.diff(f,x)+10*f); label=r'$(D+1)^2+9$'
    grid=np.linspace(-2,2,600); ff=sp.lambdify(x,f,'numpy')
    plt.plot(grid,ff(grid)); plt.title(f'{kind} forcing; annihilator {label}'); plt.xlabel('x'); plt.ylabel('f(x)'); plt.show()
    display(sp.simplify(residual))
if WIDGETS_AVAILABLE: interact(annihilator_explorer,kind=Dropdown(options=['polynomial','exponential','trigonometric'],value='polynomial'))
else: annihilator_explorer()

## Root bookkeeping picture

The complementary roots and forcing-annihilator roots combine in the higher-order homogeneous equation. Duplicated roots explain the required power of $x$.

In [ ]:
roots_L=np.array([-1+0j,-2+0j])
roots_A=np.array([0+0j,0+0j,0+0j])
plt.scatter(roots_L.real,roots_L.imag,s=90,label='roots from L(D)')
plt.scatter(roots_A.real,roots_A.imag,s=90,marker='x',label='roots from annihilator')
plt.axhline(0); plt.axvline(0); plt.xlabel('real part'); plt.ylabel('imaginary part'); plt.title('Combined characteristic-root multiset'); plt.legend(); plt.show()

## Classroom Checkpoint — Exit Check

Give a minimal annihilator for $x^3e^{-2x}\sin5x$.

> Pause here. Let students commit to an answer before running the next cell.